# Semantic Search & LLM Application
**Starter Notebook**

Notebook ini adalah kerangka awal untuk membangun prototype aplikasi semantic search yang terintegrasi dengan LLM, menggunakan knowledge base FAQ e-commerce.

Alur besar: Load Knowledge Base → Embedding → Vector Database (FAISS) → Semantic Search → Prompt Design → LLM Integration → Uji Relevansi.

Isi tiap section sesuai instruksi di Assignment Brief. Jangan ubah struktur section, tapi silakan tambah cell baru di dalam tiap section jika diperlukan.

## 1. Import Library & Load Knowledge Base

In [1]:
import pandas as pd
import numpy as np
import faiss
from google import genai  # sesuaikan dengan library Gemini API yang digunakan
from google.genai import types
import os

client = genai.Client(api_key=os.environ.get('GEMINI_API_KEY'))

# Memuat knowledge base dari folder data/ sesuai struktur proyek
data_path = '../data/ecommerce_faq_knowledge_base.csv' if os.path.exists('../data/ecommerce_faq_knowledge_base.csv') else 'data/ecommerce_faq_knowledge_base.csv'
df = pd.read_csv(data_path)
print('Total entri knowledge base:', len(df))
df.head()

Total entri knowledge base: 59


,faq_id,category,question,answer
0,1,Pemesanan,Bagaimana cara melakukan pemesanan di platform ini?,"Pilih produk yang diinginkan, klik 'Tambah ke Keranjang', lalu lanjutkan ke halaman checkout untuk mengisi alamat pengiriman dan metode pembayaran."
1,2,Pemesanan,Apakah saya bisa memesan tanpa membuat akun?,"Saat ini pemesanan hanya bisa dilakukan setelah membuat akun, agar Anda dapat melacak status pesanan dengan mudah."
2,3,Pemesanan,Bagaimana cara membatalkan pesanan yang sudah dibuat?,"Pesanan dapat dibatalkan melalui halaman 'Pesanan Saya' selama status masih 'Diproses'. Setelah status berubah menjadi 'Dikirim', pembatalan tidak dapat dilakukan."
3,4,Pemesanan,Berapa lama waktu yang dibutuhkan sebelum pesanan diproses penjual?,Penjual biasanya memproses pesanan dalam waktu 1x24 jam setelah pembayaran dikonfirmasi.
4,5,Pemesanan,Apakah saya bisa mengubah alamat pengiriman setelah pesanan dibuat?,Perubahan alamat hanya bisa dilakukan sebelum pesanan berstatus 'Dikirim'. Silakan hubungi customer service secepatnya jika ingin mengubah alamat.


## 2. Menyiapkan Unit Retrievable

Setiap entri FAQ (question + answer) diperlakukan sebagai satu unit yang akan di-embed dan disimpan ke vector database. Gabungkan question dan answer menjadi satu teks representatif per entri.

In [2]:
df['content'] = 'Q: ' + df['question'] + ' A: ' + df['answer']
documents = df['content'].tolist()
print('Contoh unit retrievable:')
print(documents[0])

Contoh unit retrievable:
Q: Bagaimana cara melakukan pemesanan di platform ini? A: Pilih produk yang diinginkan, klik 'Tambah ke Keranjang', lalu lanjutkan ke halaman checkout untuk mengisi alamat pengiriman dan metode pembayaran.


## 3. Membuat Embeddings

Ubah setiap unit retrievable menjadi vektor menggunakan Gemini Embeddings API.

In [3]:
def get_embedding(text):
    result = client.models.embed_content(
        model='gemini-embedding-001',  # sesuaikan versi model embedding (menggunakan gemini-embedding-001)
        contents=text
    )
    return result.embeddings[0].values

embeddings = [get_embedding(doc) for doc in documents]
embeddings = np.array(embeddings).astype('float32')
print('Shape embeddings:', embeddings.shape)

Shape embeddings: (59, 3072)


## 4. Menyimpan Embeddings ke Vector Database (FAISS)

In [4]:
dimension = embeddings.shape[1]
index = faiss.IndexFlatL2(dimension)
index.add(embeddings)

print('Jumlah vektor tersimpan di index:', index.ntotal)

Jumlah vektor tersimpan di index: 59


## 5. Mengimplementasikan Semantic Search

Fungsi untuk mengambil beberapa entri FAQ paling relevan berdasarkan query pengguna.

In [5]:
def semantic_search(query, top_k=3):
    query_embedding = np.array([get_embedding(query)]).astype('float32')
    distances, indices = index.search(query_embedding, top_k)

    results = []
    for rank, idx in enumerate(indices[0]):
        results.append({
            'rank': rank + 1,
            'question': df.iloc[idx]['question'],
            'answer': df.iloc[idx]['answer'],
            'category': df.iloc[idx]['category'],
            'distance': distances[0][rank]
        })
    return results

# Contoh percobaan
test_results = semantic_search('kenapa paket saya belum sampai?')
for r in test_results:
    print(r)

{'rank': 1, 'question': 'Bagaimana jika paket saya belum sampai melebihi estimasi?', 'answer': 'Jika paket melebihi estimasi pengiriman lebih dari 3 hari, silakan hubungi customer service dengan menyertakan nomor resi untuk ditelusuri lebih lanjut.', 'category': 'Pengiriman', 'distance': 0.3335350751876831}
{'rank': 2, 'question': 'Kenapa status pengiriman saya tidak berubah selama beberapa hari?', 'answer': 'Status yang tidak berubah bisa disebabkan keterlambatan update sistem kurir. Disarankan menunggu 1x24 jam sebelum melaporkan ke customer service.', 'category': 'Pengiriman', 'distance': 0.4088098108768463}
{'rank': 3, 'question': 'Kenapa nomor resi saya belum bisa dilacak di website kurir?', 'answer': "Nomor resi baru dapat dilacak setelah paket diserahterimakan ke pihak kurir, biasanya butuh waktu beberapa jam setelah status 'Dikirim'.", 'category': 'Pengiriman', 'distance': 0.43201810121536255}


## 6. Mendesain Prompt dan Context

Susun prompt terstruktur (Persona, Task, Context, Format) yang menyisipkan hasil semantic search sebagai bagian Context, sebelum dikirim ke LLM.

In [6]:
def build_prompt(query, search_results):
    context = '\n\n'.join(
        f"FAQ: {r['question']}\nJawaban: {r['answer']}" for r in search_results
    )
    prompt = f'''[Persona]
Kamu adalah asisten customer service e-commerce yang ramah dan membantu.

[Task]
Jawab pertanyaan pengguna HANYA berdasarkan informasi FAQ berikut. Jika informasi tidak ditemukan dalam FAQ, katakan bahwa kamu belum memiliki informasi tersebut dan sarankan menghubungi customer service.

[Context]
{context}

[Format]
Jawab dengan ramah, sopan, jelas, dan langsung pada inti informasi.

Pertanyaan pengguna: {query}

Jawaban:'''
    return prompt

## 7. Mengintegrasikan LLM

In [7]:
def generate_answer(query, top_k=3):
    search_results = semantic_search(query, top_k=top_k)
    prompt = build_prompt(query, search_results)

    response = client.models.generate_content(
        model='gemini-3.5-flash',  # sesuaikan versi model (menggunakan gemini-3.5-flash)
        contents=prompt,
        config=types.GenerateContentConfig(temperature=0.3)
    )
    return response.text, search_results

answer, sources = generate_answer('apakah bisa bayar pakai cicilan?')
print('Jawaban LLM:')
print(answer)
print()
print('Sumber FAQ yang digunakan:')
for s in sources:
    print('-', s['question'])

Jawaban LLM:
Halo Kak! Tentu bisa. 

Fitur cicilan tersedia untuk transaksi menggunakan kartu kredit tertentu dengan nominal minimal yang ditentukan oleh pihak bank. 

Ada lagi yang bisa saya bantu, Kak? 😊

Sumber FAQ yang digunakan:
- Apakah saya bisa membayar dengan cicilan?
- Apakah bisa membayar menggunakan lebih dari satu metode sekaligus?
- Metode pembayaran apa saja yang tersedia?


## 8. Menguji Relevansi Hasil Retrieval dan Response AI

Uji sistem dengan beberapa query berbeda, termasuk query yang menggunakan kata/frasa berbeda dari FAQ asli (sinonim/parafrase) untuk menilai apakah semantic search benar-benar menangkap makna, bukan hanya mencocokkan kata kunci.

_Uji minimal 5 query berbeda, termasuk:_
- _1-2 query yang jelas ada jawabannya di FAQ_
- _1-2 query dengan kata-kata berbeda dari FAQ asli (parafrase/sinonim)_
- _1 query yang TIDAK ada jawabannya di FAQ (uji bagaimana LLM merespons ketidaktahuannya)_

In [8]:
test_queries = [
    "Bagaimana cara membatalkan pesanan yang sudah dibuat?",
    "Berapa lama estimasi waktu pengiriman barang untuk area luar pulau?",
    "Apakah toko menerima opsi pembayaran bertahap atau sistem kredit kartu?",
    "Barang pesanan yang saya terima ternyata rusak dan cacat, bagaimana prosedur komplain dan retur?",
    "Apakah kantor e-commerce ini sedang membuka lowongan magang untuk posisi AI Engineer?"
]

for q in test_queries:
    print('Query:', q)
    answer, sources = generate_answer(q)
    print('Jawaban:', answer)
    print('Sumber teratas:', sources[0]['question'] if sources else '-')
    print('---')

Query: Bagaimana cara membatalkan pesanan yang sudah dibuat?
Jawaban: Halo! Anda dapat membatalkan pesanan melalui halaman **'Pesanan Saya'** selama status pesanan Anda masih **'Diproses'**. 

Mohon diingat, jika status pesanan sudah berubah menjadi **'Dikirim'**, maka pembatalan sudah tidak dapat dilakukan lagi ya.

Semoga informasi ini membantu! Jika ada hal lain yang ingin ditanyakan, jangan ragu untuk memberi tahu saya.
Sumber teratas: Bagaimana cara membatalkan pesanan yang sudah dibuat?
---
Query: Berapa lama estimasi waktu pengiriman barang untuk area luar pulau?
Jawaban: Halo! Untuk estimasi waktu pengiriman barang ke area luar pulau adalah 5-10 hari kerja, tergantung pada jasa kirim yang dipilih. 

Semoga informasi ini membantu! Jika ada pertanyaan lain, jangan ragu untuk bertanya ya.
Sumber teratas: Berapa lama estimasi waktu pengiriman barang?
---
Query: Apakah toko menerima opsi pembayaran bertahap atau sistem kredit kartu?
Jawaban: Halo! Terima kasih telah bertanya. 

Ya, 

### Catatan Analisis Relevansi

Berdasarkan hasil pengujian 5 skenario query di atas, berikut adalah pengamatan dan analisis mengenai performa sistem:

#### 1. Query yang Jelas Ada Jawabannya di Knowledge Base (Query 1 & 2)
* **Query 1:** *"Bagaimana cara membatalkan pesanan yang sudah dibuat?"*
  * **Hasil Retrieval:** Sangat akurat. Entri FAQ teratas yang diambil adalah FAQ ID 3 (*"Bagaimana cara membatalkan pesanan yang sudah dibuat?"*) dengan nilai distance yang sangat rendah ($L_2 \approx 0.2392$).
  * **Kualitas Jawaban LLM:** Sangat relevan dan tepat. LLM menjelaskan bahwa pesanan dapat dibatalkan melalui menu 'Pesanan Saya' selama status masih 'Diproses', dan tidak bisa dibatalkan jika sudah 'Dikirim'.
* **Query 2:** *"Berapa lama estimasi waktu pengiriman barang untuk area luar pulau?"*
  * **Hasil Retrieval:** Sangat akurat. Sumber teratas adalah FAQ ID 9 (*"Berapa lama estimasi waktu pengiriman barang?"*) dengan distance $\approx 0.2212$.
  * **Kualitas Jawaban LLM:** LLM secara spesifik mengekstrak informasi yang ditanyakan, yaitu estimasi 5–10 hari kerja untuk area luar pulau, tanpa mencampuradukkan dengan estimasi dalam kota.

#### 2. Query Parafrase / Sinonim (Query 3 & 4)
* **Query 3:** *"Apakah toko menerima opsi pembayaran bertahap atau sistem kredit kartu?"*
  * **Hasil Retrieval:** Berhasil menangkap makna. Meskipun pengguna tidak menggunakan kata *"cicilan"*, semantic search berhasil memetakan konsep *"pembayaran bertahap / sistem kredit"* ke FAQ ID 20 (*"Apakah saya bisa membayar dengan cicilan?"*) dengan distance $\approx 0.4325$.
  * **Kualitas Jawaban LLM:** LLM mengonfirmasi bahwa pembayaran kartu kredit dan fitur cicilan tersedia sesuai ketentuan nominal minimal dari bank terkait.
* **Query 4:** *"Barang pesanan yang saya terima ternyata rusak dan cacat, bagaimana prosedur komplain dan retur?"*
  * **Hasil Retrieval:** Akurat dan komprehensif. Retrieval berhasil mengambil FAQ ID 24 (prosedur pengembalian barang), FAQ ID 54 (menerima barang salah), dan FAQ ID 16 (paket dalam kondisi rusak) dengan jarak yang rapat ($0.35 - 0.36$).
  * **Kualitas Jawaban LLM:** LLM menyintesis informasi tersebut menjadi langkah-langkah praktis: mengambil bukti foto/video, melapor dalam 2x24 jam, dan mengajukan retur di menu 'Pesanan Saya' maksimal 3 hari.

#### 3. Query yang Tidak Memiliki Jawaban di Knowledge Base (Query 5)
* **Query 5:** *"Apakah kantor e-commerce ini sedang membuka lowongan magang untuk posisi AI Engineer?"*
  * **Hasil Retrieval:** Nilai distance sangat tinggi ($L_2 \approx 0.8598$), menandakan tidak ada dokumen yang mirip secara semantik.
  * **Respon Ketidaktahuan LLM (Anti-Halusinasi):** Berkat arahan tegas pada prompt (`[Task]` & `[Persona]`), LLM tidak mengarang informasi dan secara jujur menyatakan bahwa informasi tersebut belum tersedia di knowledge base, serta menyarankan pengguna untuk menghubungi customer service resmi.